# 5. The configuration model: from neighbour support to a prediction

The triangle model builds connections in groups of three. To understand what this
structure changes, we compare it with a graph that has the same **limiting degree
distribution**, but pairs its edges without forming prescribed communities.

This notebook derives the configuration-model prediction for the fraction of
vertices in the 3-core. It also defines the numerical functions used in
`06_configuration_simulation.ipynb`, which runs the simulations, and
`07_configuration_analysis.ipynb`, which draws the comparison figures.
Loading this notebook only defines functions; it does not
simulate graphs or write files.

The main distinction is between two parameters:

- $\lambda$ is the mean number of triangle memberships per individual.
- $\mu=2\lambda$ is the limiting mean **graph degree** in both models.

Keeping these separate matters: this comparison is not with an Erdős–Rényi
graph of Poisson degree. Our configuration-model degree is twice a Poisson
random variable.


## Which degree distribution should we match?

An individual with $D$ triangle memberships usually meets two different
neighbours in each membership. In a finite graph, repeated individuals within
a community or overlaps between communities can reduce its number of distinct
neighbours. With fixed $\lambda$ and growing $n$, these coincidences affect
a vanishing fraction of vertices. The limiting degree is therefore

$$
K=2D,\qquad D\sim\operatorname{Poisson}(\lambda),\qquad
\mu=\mathbb{E}[K]=2\lambda.
$$

Its probability mass function is

$$
p_{2j}=e^{-\lambda}\frac{\lambda^j}{j!},\qquad p_{2j+1}=0.
$$

Matching only the mean would miss the even-valued distribution and its
variance, $\operatorname{Var}(K)=4\lambda$.

This specialization agrees with equation (2.20) and Corollary 2.9 of
[van der Hofstad, Komjáthy and Vadon, *Random intersection graphs with
communities*](https://arxiv.org/pdf/1809.02514#page=9):
the limiting degree is the sum of the internal degrees contributed by all
memberships. For a triangle, every internal degree is two. Their local limit
retains communities; it is not an ordinary tree.

Our triangle sampler conditions the whole membership vector on its sum being
divisible by three. At fixed positive $\lambda$, this global constraint does
not change the limiting degree law. The comparison matches that law, rather
than promising identical degree sequences in each pair of finite graphs.


## Why an edge can be studied through neighbour support

To construct a configuration model with degrees $K_1,\ldots,K_n$, give vertex
$i$ exactly $K_i$ half-edges and pair all half-edges uniformly. The sum must
be even; our degrees are already even. A pair can form a self-loop, and several
pairs can connect the same two vertices.

For fixed $\lambda$, exploring a bounded neighbourhood is unlikely to visit
the same vertex twice as $n$ grows. The limiting neighbourhood is a branching
tree, so support arriving along different branches becomes independent. This is
the reason a scalar message equation works here. It would fail for the triangle
graph itself: two neighbours in one triangle are connected and can support one
another.

Imagine temporarily removing one edge from a vertex. The vertex sends a
**support message** along that edge if at least two of its other neighbours
can support it. If the removed neighbour also supports the vertex, the vertex
then has the three neighbours required for the 3-core.

Let $x$ be the probability of such a message. Following a uniformly chosen
edge does not reach a uniformly chosen vertex. A degree-$k$ vertex is $k$
times as likely to be reached, giving the size-biased distribution

$$
\mathbb{P}(K^*=k)=\frac{kp_k}{\mu}.
$$

The reached vertex has $k-1$ other neighbours. In the limiting tree, the number
that support it is $\operatorname{Binomial}(k-1,x)$. Therefore

$$
\boxed{\quad
x=F(x)=\sum_{k\ge1}\frac{kp_k}{\mu}
       \mathbb{P}\!\left(\operatorname{Binomial}(k-1,x)\ge2\right).
\quad}
$$

For a uniformly chosen vertex, all $k$ neighbours are available. It remains
in the core when at least three support it, so the predicted density is

$$
\boxed{\quad
\rho(x)=\sum_{k\ge0}p_k
       \mathbb{P}\!\left(\operatorname{Binomial}(k,x)\ge3\right).
\quad}
$$

There may be several fixed points, including $x=0$. The relevant one is the
**largest** fixed point. One way to understand this is to start with every
message present ($x_0=1$) and repeatedly remove messages that lack support.
The decreasing sequence $x_{t+1}=F(x_t)$ reaches the largest fixed point.
Starting at zero would stay at zero and miss an existing core.


## How this connects to the cited theorem

[Janson and Łuczak, *A simple solution to the $k$-core problem*,
Condition 2.2 and Theorem 2.3](https://arxiv.org/pdf/math/0508453#page=4)
use a thinned degree $K_x\mid K\sim\operatorname{Binomial}(K,x)$.
For the 3-core, write

$$
h(x)=\mathbb{E}[K_x\,\mathbf 1_{\{K_x\ge3\}}],
\qquad h_1(x)=\mathbb{P}(K_x\ge3).
$$

The binomial identity

$$
\mathbb{E}[\operatorname{Binomial}(k,x)\,
             \mathbf 1_{\{\operatorname{Binomial}(k,x)\ge3\}}]
=kx\,\mathbb{P}(\operatorname{Binomial}(k-1,x)\ge2)
$$

gives $h(x)=\mu xF(x)$. Thus their equation $\mu x^2=h(x)$
is exactly our support equation for $x>0$, and $h_1(x)=\rho(x)$.

The theorem assumes convergence of the empirical degree distribution and its
mean. These hold for our independent twice-Poisson degrees; all their moments
are finite. When the largest positive root has $h(x)>\mu x^2$ just below it,
the multigraph core fraction converges in probability to $h_1(x)$.
When no positive root exists, the limiting fraction is zero.
The tangency at the critical parameter does **not** satisfy that crossing
condition, so this statement does not assign a deterministic core fraction
at criticality.

Our simulation retains the **CM multigraph**, counting loops twice in degree
and retaining parallel edges. This is the direct comparison with Theorem 2.3.
It also records a **CM erased** control: the very same pairing after removing
loops and merging repeated edges, so degree counts distinct neighbours as in
the triangle graph. These two observations are paired, not independent.

Theorem 2.4 treats a uniformly sampled simple graph with prescribed degrees
under additional moment conditions, satisfied here. Erasing a multigraph is
a different finite-$n$ distribution from conditioning on simplicity. Defects
are sparse at fixed $\lambda$, but deleting even a few edges can cause a
pruning cascade near the transition. Notebook 06 records these defects, and
notebook 07 measures their effects. The theorem does not provide an exact
formula for the erased control or for any particular finite graph.


## Work out the formulas for twice-Poisson degrees

Probability generating functions make the binomial sums short. Define

$$
G_0(z)=\mathbb{E}[z^K]=e^{\lambda(z^2-1)},\qquad
G_1(z)=\mathbb{E}[z^{K^*-1}]
      =\frac{G_0'(z)}{\mu}=z\,e^{\lambda(z^2-1)}
$$

for $\lambda>0$. Equivalently, $K^*-1$ has distribution
$1+2\operatorname{Poisson}(\lambda)$. The extra one is an ordinary
remaining half-edge; it is not a prescribed triangle connection.

For any nonnegative integer $L$ with generating function $G$,
the probabilities of zero and one supporting neighbour are $G(1-x)$
and $xG'(1-x)$. Applying this to $L=K^*-1$ gives

$$
F_\lambda(x)=1-G_1(1-x)-xG_1'(1-x).
$$

Since $G_1'(z)=e^{\lambda(z^2-1)}(1+2\lambda z^2)$, this simplifies to

$$
\boxed{
F_\lambda(x)=1-e^{-\lambda x(2-x)}
                    \bigl[1+2\lambda x(1-x)^2\bigr].
}
$$

For a vertex to remain in the 3-core, exclude zero, one, and two supporting
neighbours:

$$
\rho_\lambda(x)=1-G_0(1-x)-xG_0'(1-x)
                    -\frac{x^2}{2}G_0''(1-x).
$$

Using $G_0'(z)=2\lambda zG_0(z)$ and
$G_0''(z)=(2\lambda+4\lambda^2z^2)G_0(z)$, we obtain

$$
\boxed{
\rho_\lambda(x)=1-e^{-\lambda x(2-x)}
 \bigl[1+2\lambda x(1-x)+\lambda x^2
                 +2\lambda^2x^2(1-x)^2\bigr].
}
$$

These are our specialization of the general equations, rather than formulas
quoted directly from the paper. At $\lambda=0$, the graph has no edges and
the predicted density is zero. The code also returns zero for $F_0$:
there is then no edge along which to define a size-biased vertex.


In [ ]:
import numpy as np


def _cm_inputs(x, lam):
    """Check the probability x and the nonnegative membership mean lambda."""
    x = np.asarray(x, dtype=float)
    lam = float(lam)
    if not np.isfinite(lam) or lam < 0:
        raise ValueError("lambda must be finite and nonnegative.")
    if not np.all(np.isfinite(x)) or np.any((x < 0) | (x > 1)):
        raise ValueError("The support probability x must lie in [0, 1].")
    return x, lam


def cm_message(x, lam):
    """Probability of at least two other supports; x may be a NumPy array."""
    x, lam = _cm_inputs(x, lam)
    exponent = lam * x * (2 - x)
    # expm1 evaluates exp(t) - 1 accurately when t is close to zero.
    result = -np.expm1(-exponent) - (2 * lam * x * (1 - x) ** 2 * np.exp(-exponent))
    result = np.clip(result, 0.0, 1.0)
    return float(result) if result.ndim == 0 else result


def cm_density(x, lam):
    """Fraction with at least three supports at a specified support probability."""
    x, lam = _cm_inputs(x, lam)
    exponent = lam * x * (2 - x)
    one_or_two = 2 * lam * x * (1 - x) + lam * x**2 + 2 * lam**2 * x**2 * (1 - x) ** 2
    result = -np.expm1(-exponent) - one_or_two * np.exp(-exponent)
    result = np.clip(result, 0.0, 1.0)
    return float(result) if result.ndim == 0 else result


def cm_message_derivative(x, lam):
    """Derivative of F with respect to x, used to locate its tangency."""
    x, lam = _cm_inputs(x, lam)
    result = (
        2
        * lam
        * x
        * (1 - x)
        * np.exp(-lam * x * (2 - x))
        * (3 + 2 * lam * (1 - x) ** 2)
    )
    return float(result) if result.ndim == 0 else result

## Locate the transition without missing a narrow branch

For every finite $\lambda$, $F_\lambda(0)=0$ and
$F_\lambda'(0)=0$. In fact,

$$
F_\lambda(x)=(3\lambda+2\lambda^2)x^2+O(x^3).
$$

A positive root therefore does not grow continuously from the zero root at a
finite parameter. Instead, the first positive roots appear when the curve
$F_\lambda(x)$ touches the line $x$ at an interior point:

$$
F_{\lambda_c}(x_c)=x_c,\qquad
F_{\lambda_c}'(x_c)=1.
$$

The derivative needed below is

$$
F_\lambda'(x)=2\lambda x(1-x)e^{-\lambda x(2-x)}
                        \bigl[3+2\lambda(1-x)^2\bigr].
$$

Here is a convenient way to find that touching point.

1. For a proposed $0<x<1$, solve $F_\lambda(x)=x$ for $\lambda$.
   The solution is unique because $F_\lambda(x)$ increases strictly with
   $\lambda$, from zero to one. Call it $\Lambda(x)$.
2. Find the minimum of $\Lambda(x)$. It diverges near both ends of $(0,1)$,
   so its minimum is in the interior.
3. Refine the bracket by solving the tangency condition. Implicit
   differentiation gives

   $$
   \Lambda'(x)=\frac{1-F_{\Lambda(x)}'(x)}
                    {\partial_\lambda F_\lambda(x)|_{\lambda=\Lambda(x)}}.
   $$

   Along this branch, the message derivative $F_{\Lambda(x)}'(x)$ changes
   from above one to below one across the minimum; $\Lambda'(x)$ changes
   from negative to positive.

The initial grid below only brackets the minimum. Both the degree parameter and
the tangency are then found by bisection on continuous functions. We do not use
the grid as the threshold estimate or hope that it hits two closely spaced
roots.

The result is approximately

$$
\lambda_c=1.28772037,\qquad x_c=0.43903221,\qquad
\rho_c=\rho_{\lambda_c}(x_c)=0.13788881.
$$

The limiting prediction thus jumps from zero below the threshold to a positive
right-hand limit of about $13.8\%$. The corresponding mean degree is
$2\lambda_c\approx2.57544$. This is a prediction for twice-Poisson degrees,
not the familiar Poisson-degree Erdős–Rényi threshold.


In [ ]:
def _cm_bisect(function, left, right, tolerance=1e-15):
    """Solve a sign-bracketed scalar equation by repeatedly halving the interval."""
    left_value = function(left)
    right_value = function(right)
    if left_value == 0:
        return float(left)
    if right_value == 0:
        return float(right)
    if left_value * right_value > 0:
        raise ValueError("Bisection needs opposite signs at the interval ends.")
    for _ in range(120):
        middle = (left + right) / 2
        middle_value = function(middle)
        if middle_value == 0 or right - left <= tolerance * max(1, abs(middle)):
            return float(middle)
        if left_value * middle_value > 0:
            left, left_value = middle, middle_value
        else:
            right = middle
    raise RuntimeError("Bisection did not reach the requested accuracy.")


def _cm_lambda_for_message(x):
    """Find the unique lambda for which a chosen positive x is a fixed point."""
    if not 0 < x < 1:
        raise ValueError("A positive fixed-point branch requires 0 < x < 1.")
    upper = 1.0
    while cm_message(x, upper) < x:
        upper *= 2
    return _cm_bisect(lambda lam: cm_message(x, lam) - x, 0.0, upper)


def cm_critical_point():
    """Find the first positive branch, returning its location and residuals."""
    # This grid brackets the interior minimum; it does not set its precision.
    messages = np.linspace(0.001, 0.999, 129)
    lambdas = np.array([_cm_lambda_for_message(x) for x in messages])
    minimum = int(np.argmin(lambdas))
    if minimum in (0, len(messages) - 1):
        raise RuntimeError("The critical-point minimum was not bracketed.")

    def tangent_residual(x):
        lam = _cm_lambda_for_message(x)
        return cm_message_derivative(x, lam) - 1

    x = _cm_bisect(tangent_residual, messages[minimum - 1], messages[minimum + 1])
    lam = _cm_lambda_for_message(x)
    return {
        "lambda": lam,
        "x": x,
        "rho": cm_density(x, lam),
        "residual": abs(cm_message(x, lam) - x),
        "tangent_residual": abs(cm_message_derivative(x, lam) - 1),
    }

## Select the largest root and calculate the predicted density

Below $\lambda_c$, the prediction is zero. Above it, the largest root lies
between $x_c$ and one. This gives a reliable bracket even immediately above
the transition, when the two positive roots are too close for an ordinary
grid search to distinguish.

Bisection also avoids the slow convergence of repeated message updates near
a tangency. The function reports its fixed-point residual so the numerical
answer can be checked separately from the simulation.

At exactly $\lambda_c$, we return $(x_c,\rho_c)$ as the **right-hand branch
limit**. This is a plotting convention for the limiting curve. It is not a
claim that a finite graph at that parameter has this deterministic core
fraction. Notebook 07 draws the jump explicitly and treats the critical
window as a finite-size question.

For repeated predictions, calculate the critical point once and pass it in:
```python
critical = cm_critical_point()
predictions = [cm_prediction(lam, critical) for lam in lambdas]
```


In [ ]:
def cm_prediction(lam, critical=None):
    """Largest-root prediction; the critical value denotes its right-hand limit."""
    _, lam = _cm_inputs(0.0, lam)
    if critical is None:
        critical = cm_critical_point()
    if lam < critical["lambda"]:
        return {"lambda": lam, "x": 0.0, "rho": 0.0, "residual": 0.0}

    # Differences of a few floating-point units cannot resolve the double root.
    if lam - critical["lambda"] <= 8 * np.spacing(critical["lambda"]):
        x = critical["x"]
    else:
        x = _cm_bisect(
            lambda value: cm_message(value, lam) - value,
            critical["x"],
            1.0,
        )
    return {
        "lambda": lam,
        "x": x,
        "rho": cm_density(x, lam),
        "residual": abs(cm_message(x, lam) - x),
    }

## Checks to carry into the numerical comparison

The formulas can be checked independently by summing over
$D\sim\operatorname{Poisson}(\lambda)$. For $j\ge1$, the contribution to
the message equation is

$$
\frac{j\,\mathbb{P}(D=j)}{\lambda}
  \mathbb{P}(\operatorname{Binomial}(2j-1,x)\ge2),
$$

and the contribution to the density is

$$
\mathbb{P}(D=j)
  \mathbb{P}(\operatorname{Binomial}(2j,x)\ge3).
$$

These sums agree with the generating-function formulas. The automated checks
evaluate both expressions independently, check the two tangency equations,
and examine fixed points just below and above the transition.

Notebook 07 overlays this limiting curve on independently sampled
configuration-model graphs at several sizes. Away from the transition,
agreement should improve as finite-size effects decrease. Near the transition,
some trials may have little core and others a substantial one; their mean
need not resemble the discontinuous limiting curve.

Finally, matching a degree distribution does not match all other structure.
The triangle model has persistent short cycles and dependent neighbour
support, whereas the configuration model pairs half-edges freely. Differences
are evidence about these structural changes together. The scalar equation in
this notebook is not a theoretical prediction for the triangle model.
